In [ ]:
pip -q install -U autoawq transformers accelerate

In [ ]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, AutoConfig
import gc
import os

# Configure environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Checking PyTorch and CUDA versions...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name()}")

# Clear any existing cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(
    base_model,
    use_fast=True,
    trust_remote_code=True
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading model...")
# Load model with specific configurations to avoid attention issues
mdl = AutoAWQForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.float16,  # Use fp16 explicitly
    device_map={"": 0} if torch.cuda.is_available() else "cpu"  # More specific device mapping
)

# Set model to eval mode
mdl.eval()

# AWQ quantization config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# Create more diverse and shorter calibration data
print("Preparing calibration data...")
calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models process data efficiently.",
    "Natural language understanding is advancing rapidly.",
    "Deep neural networks learn complex patterns.",
    "Artificial intelligence transforms technology.",
    "Computer vision recognizes objects accurately.",
    "Robotics integrates sensors and actuators.",
    "Algorithm optimization improves performance significantly.",
    "Data science extracts meaningful insights.",
    "Software engineering creates reliable systems."
] * 10  # 100 samples total

print("Starting quantization...")
try:
    # Use minimal configuration to avoid batch size issues
    mdl.quantize(
        tok,
        quant_config=quant_config,
        calib_data=calib_texts,
        max_calib_seq_len=128,     # Reduced sequence length
        max_calib_samples=50,      # Reduced sample count
        n_parallel_calib_samples=1 # Keep sequential processing
    )

    print("Quantization completed successfully!")

    # Save the quantized model
    out_dir = "tinyllama-1.1b-awq"
    print(f"Saving model to {out_dir}...")

    mdl.save_quantized(out_dir, safetensors=True)
    tok.save_pretrained(out_dir)

    # Save config
    config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
    config.save_pretrained(out_dir)

    print(f"Model successfully quantized and saved to {out_dir}")

except Exception as e:
    print(f"Quantization failed with error: {str(e)}")
    print("\nTrying alternative approach with different model loading...")

    # Alternative approach: Load model differently
    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    try:
        # Try loading without device_map first
        mdl = AutoAWQForCausalLM.from_pretrained(
            base_model,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_cache=False
        )

        # Move to device manually if CUDA is available
        if torch.cuda.is_available():
            mdl = mdl.cuda()

        mdl.eval()

        # Try with even smaller calibration parameters
        print("Attempting quantization with minimal parameters...")
        mdl.quantize(
            tok,
            quant_config=quant_config,
            calib_data=calib_texts[:20],  # Only use first 20 samples
            max_calib_seq_len=64,         # Even smaller sequence length
            max_calib_samples=20,         # Minimal samples
            n_parallel_calib_samples=1
        )

        out_dir = "tinyllama-1.1b-awq"
        mdl.save_quantized(out_dir, safetensors=True)
        tok.save_pretrained(out_dir)

        config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
        config.save_pretrained(out_dir)

        print(f"Model successfully quantized with alternative approach and saved to {out_dir}")

    except Exception as e2:
        print(f"Alternative approach also failed: {str(e2)}")
        print("\nConsider using a pre-quantized model instead:")
        print("TheBloke/TinyLlama-1.1B-Chat-v1.0-AWQ")

finally:
    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

In [ ]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer, AutoConfig
import gc
import os

# Configure environment
os.environ["TOKENIZERS_PARALLELISM"] = "false"

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Checking PyTorch and CUDA versions...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name()}")

# Clear any existing cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(
    base_model,
    use_fast=True,
    trust_remote_code=True
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading model...")
# Load model with specific configurations to avoid attention issues
mdl = AutoAWQForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    use_cache=False,
    trust_remote_code=True,
    torch_dtype=torch.float16,  # Use fp16 explicitly
    device_map={"": 0} if torch.cuda.is_available() else "cpu"  # More specific device mapping
)

# Set model to eval mode
mdl.eval()

# AWQ quantization config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

# Create more diverse and shorter calibration data
print("Preparing calibration data...")
calib_texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models process data efficiently.",
    "Natural language understanding is advancing rapidly.",
    "Deep neural networks learn complex patterns.",
    "Artificial intelligence transforms technology.",
    "Computer vision recognizes objects accurately.",
    "Robotics integrates sensors and actuators.",
    "Algorithm optimization improves performance significantly.",
    "Data science extracts meaningful insights.",
    "Software engineering creates reliable systems."
] * 10  # 100 samples total

print("Starting quantization...")
try:
    # Use minimal configuration to avoid batch size issues
    mdl.quantize(
        tok,
        quant_config=quant_config,
        calib_data=calib_texts,
        max_calib_seq_len=128,     # Reduced sequence length
        max_calib_samples=50,      # Reduced sample count
        n_parallel_calib_samples=1 # Keep sequential processing
    )

    print("Quantization completed successfully!")

    # Save the quantized model
    out_dir = "tinyllama-1.1b-awq"
    print(f"Saving model to {out_dir}...")

    mdl.save_quantized(out_dir, safetensors=True)
    tok.save_pretrained(out_dir)

    # Save config
    config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
    config.save_pretrained(out_dir)

    print(f"Model successfully quantized and saved to {out_dir}")

except Exception as e:
    print(f"Quantization failed with error: {str(e)}")
    print("\nTrying alternative approach with different model loading...")

    # Alternative approach: Load model differently
    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    try:
        # Try loading without device_map first
        mdl = AutoAWQForCausalLM.from_pretrained(
            base_model,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_cache=False
        )

        # Move to device manually if CUDA is available
        if torch.cuda.is_available():
            mdl = mdl.cuda()

        mdl.eval()

        # Try with even smaller calibration parameters
        print("Attempting quantization with minimal parameters...")
        mdl.quantize(
            tok,
            quant_config=quant_config,
            calib_data=calib_texts[:20],  # Only use first 20 samples
            max_calib_seq_len=64,         # Even smaller sequence length
            max_calib_samples=20,         # Minimal samples
            n_parallel_calib_samples=1
        )

        out_dir = "tinyllama-1.1b-awq"
        mdl.save_quantized(out_dir, safetensors=True)
        tok.save_pretrained(out_dir)

        config = AutoConfig.from_pretrained(base_model, trust_remote_code=True)
        config.save_pretrained(out_dir)

        print(f"Model successfully quantized with alternative approach and saved to {out_dir}")

    except Exception as e2:
        print(f"Alternative approach also failed: {str(e2)}")
        print("\nConsider using a pre-quantized model instead:")
        print("TheBloke/TinyLlama-1.1B-Chat-v1.0-AWQ")

finally:
    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()